# MedVisionAI — Diabetic Retinopathy Screening Prototype
## Kaggle Training & Evaluation Notebook (EfficientNet-B0)

**Project:** MedVisionAI — Low-cost AI-assisted medical image screening system  
**Task:** Binary Diabetic Retinopathy (DR) Screening from Retinal Fundus Photographs  
**Model Architecture:** Pretrained `EfficientNet-B0` (Transfer Learning)  
**Dataset:** APTOS 2019 Blindness Detection  

---
### 1. Environment Verification
Verify PyTorch version, CUDA availability, GPU device info, and working directory setup.

In [ ]:
import torch
import torchvision
import os
import sys
import json
import zipfile
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# Check hardware and environment
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Device: {gpu_name}")
    print(f"CUDA Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: GPU not available. Running on CPU.")

print(f"Current Working Directory: {os.getcwd()}")

---
### 2. Dataset Discovery
Dynamically search `/kaggle/input/` to locate `train.csv` and the retinal images directory (`train_images/`).

In [ ]:
def discover_dataset_paths():
    base_input = "/kaggle/input"
    csv_path = None
    images_dir = None
    
    if not os.path.exists(base_input):
        print(f"Directory '{base_input}' not found. Checking local workspace directory...")
        base_input = "."
        
    for root, dirs, files in os.walk(base_input):
        if "train.csv" in files:
            csv_path = os.path.join(root, "train.csv")
        if "train_images" in dirs:
            images_dir = os.path.join(root, "train_images")
            
    if csv_path is None or images_dir is None:
        raise FileNotFoundError(
            f"Could not locate train.csv or train_images/ in {base_input}. "
            "Please ensure the APTOS 2019 dataset is attached to the Kaggle notebook."
        )
        
    return csv_path, images_dir

CSV_PATH, IMAGES_DIR = discover_dataset_paths()
print(f"Discovered train.csv at: {CSV_PATH}")
print(f"Discovered train_images at: {IMAGES_DIR}")

---
### 3. Dataset Exploration
Load `train.csv` and inspect shape, columns, head rows, missing values, and the original 5-class severity distribution (0 = No DR, 1 = Mild, 2 = Moderate, 3 = Severe, 4 = Proliferative DR).

In [ ]:
df = pd.read_csv(CSV_PATH)
print("=== Dataset Summary ===")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"Missing Values:\n{df.isnull().sum()}")

print("\nFirst 10 Rows:")
print(df.head(10))

print("\nOriginal 5-Class Severity Distribution:")
diagnosis_counts = df['diagnosis'].value_counts().sort_index()
class_names_5 = {
    0: "No DR",
    1: "Mild DR",
    2: "Moderate DR",
    3: "Severe DR",
    4: "Proliferative DR"
}

for code, name in class_names_5.items():
    count = diagnosis_counts.get(code, 0)
    pct = (count / len(df)) * 100
    print(f"  Class {code} ({name}): {count} images ({pct:.2f}%)")

# Plot original class distribution
plt.figure(figsize=(8, 4))
sns.barplot(x=[class_names_5[c] for c in diagnosis_counts.index], y=diagnosis_counts.values, palette="viridis")
plt.title("APTOS 2019 Original Severity Distribution")
plt.xlabel("Severity Grade")
plt.ylabel("Number of Images")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

---
### 4. Image Visualization
Randomly display retinal fundus images alongside their clinician-assigned severity and binary classification labels to inspect image quality.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample_df = df.sample(8, random_state=42).reset_index(drop=True)

for i, row in sample_df.iterrows():
    ax = axes[i // 4, i % 4]
    img_name = f"{row['id_code']}.png"
    img_path = os.path.join(IMAGES_DIR, img_name)
    
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img)
        diag = row['diagnosis']
        binary_lbl = "DR PRESENT" if diag > 0 else "NO DR"
        ax.set_title(f"ID: {row['id_code']}\nDiag: {diag} ({class_names_5[diag]})\nBinary: {binary_lbl}", fontsize=10)
    else:
        ax.text(0.5, 0.5, "Image Not Found", ha='center')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
### 5. Binary Label Conversion
Convert the multi-class problem into a binary screening task:
* `Original label 0` $\rightarrow$ `0 = NO DR`
* `Original labels 1, 2, 3, 4` $\rightarrow$ `1 = DR PRESENT`

In [ ]:
df['label'] = (df['diagnosis'] > 0).astype(int)

binary_counts = df['label'].value_counts().sort_index()
class_names_binary = {0: "No DR", 1: "DR Present"}

print("=== Binary Class Distribution ===")
for code, name in class_names_binary.items():
    count = binary_counts.get(code, 0)
    pct = (count / len(df)) * 100
    print(f"  Class {code} ({name}): {count} images ({pct:.2f}%)")

plt.figure(figsize=(6, 4))
sns.barplot(x=["No DR (0)", "DR Present (1)"], y=binary_counts.values, palette=["#2ecc71", "#e74c3c"])
plt.title("Binary Target Distribution (No DR vs DR Present)")
plt.ylabel("Number of Images")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

---
### 6. Stratified Train / Validation / Test Split
Split the dataset into:
* **70% Training**
* **15% Validation**
* **15% Test (Held-out unseen evaluation set)**

Using stratified splitting based on `label` with `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

# 70% Train, 30% Temp (Val + Test)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=42
)

# Split 30% Temp evenly into 15% Val and 15% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

print(f"Training set:   {len(train_df)} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation set: {len(val_df)} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set:       {len(test_df)} images ({len(test_df)/len(df)*100:.1f}%)")

print("\nClass Distributions:")
print("  Train:", dict(train_df['label'].value_counts()))
print("  Val:  ", dict(val_df['label'].value_counts()))
print("  Test: ", dict(test_df['label'].value_counts()))

---
### 7. Image Preprocessing & Data Augmentations
Define PyTorch data transformations for EfficientNet-B0 (224x224 input):
* **Training Transforms:** Random horizontal flip, small random rotation ($\pm 15^\circ$), moderate brightness & contrast jitter, tensor conversion, and ImageNet normalization.
* **Validation/Test Transforms:** Deterministic resizing to 224x224, tensor conversion, and ImageNet normalization.

In [ ]:
from torchvision import transforms

# ImageNet normalization values required for pretrained torchvision models
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

---
### 8. PyTorch Dataset & DataLoader Setup
Implement a custom PyTorch `Dataset` to load fundus images on demand and create PyTorch `DataLoader` objects.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class APTOSDataset(Dataset):
    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = f"{row['id_code']}.png"
        img_path = os.path.join(self.images_dir, img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label

BATCH_SIZE = 32

train_dataset = APTOSDataset(train_df, IMAGES_DIR, transform=train_transforms)
val_dataset   = APTOSDataset(val_df, IMAGES_DIR, transform=val_test_transforms)
test_dataset  = APTOSDataset(test_df, IMAGES_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train Batches: {len(train_loader)} (batch_size={BATCH_SIZE})")
print(f"Val Batches:   {len(val_loader)}")
print(f"Test Batches:  {len(test_loader)}")

---
### 9. EfficientNet-B0 Model Architecture
Initialize `EfficientNet-B0` with pretrained ImageNet weights and replace the final linear layer with a 2-class binary head (`0 = No DR`, `1 = DR Present`).

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def get_model():
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)
    
    # Replace final classifier layer
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 2)
    
    return model

model = get_model().to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== Model Architecture Setup ===")
print(f"Model Backbone: EfficientNet-B0")
print(f"Pretrained Weights: ImageNet (DEFAULT)")
print(f"Output Classes: 2 (No DR / DR Present)")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

---
### 10. Class Imbalance Handling
Calculate inverse class weights to handle slight class imbalance in the training set and instantiate weighted `CrossEntropyLoss`.

In [ ]:
# Calculate inverse class frequency weights for training set
class_counts = train_df['label'].value_counts().sort_index().values
total_samples = len(train_df)

# Weight formula: N / (num_classes * count_i)
class_weights = total_samples / (2.0 * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Train Class Counts (0 vs 1): {class_counts}")
print(f"Calculated Loss Class Weights: No DR={class_weights[0]:.4f}, DR={class_weights[1]:.4f}")

# Weighted Cross Entropy Loss
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

---
### 11. Model Training & Validation Loop
Train EfficientNet-B0 using AdamW optimizer (`lr=1e-4`, `weight_decay=1e-2`), save best validation checkpoint (`dr_efficientnet_b0_best.pth`), and implement early stopping.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score
import time

EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_f1': [], 'val_auc': []
}

best_val_loss = float('inf')
CHECKPOINT_PATH = "/kaggle/working/dr_efficientnet_b0_best.pth"
if not os.path.exists("/kaggle/working"):
    os.makedirs("/kaggle/working", exist_ok=True)

print("Starting Training...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    # Training Phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # Validation Phase
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    val_preds_list = []
    val_probs_list = []
    val_labels_list = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            
            val_preds_list.extend(preds.cpu().numpy())
            val_probs_list.extend(probs[:, 1].cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_running_loss / total_val
    epoch_val_acc = correct_val / total_val
    epoch_val_f1 = f1_score(val_labels_list, val_preds_list, zero_division=0)
    epoch_val_auc = roc_auc_score(val_labels_list, val_probs_list)
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_f1'].append(epoch_val_f1)
    history['val_auc'].append(epoch_val_auc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}% | "
          f"Val F1: {epoch_val_f1:.4f} | Val AUC: {epoch_val_auc:.4f}")
    
    # Save Best Model Checkpoint
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  --> Saved new best checkpoint to {CHECKPOINT_PATH} (Val Loss: {epoch_val_loss:.4f})")

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time // 60:.0f}m {total_time % 60:.0f}s.")

---
### 12. Training History Curves
Plot Loss and Accuracy curves for Training vs Validation sets over epochs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss Plot
axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#2980b9')
axes[0].plot(epochs_range, history['val_loss'], 's-', label='Val Loss', color='#e74c3c')
axes[0].set_title('Training vs Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# Accuracy Plot
axes[1].plot(epochs_range, [acc * 100 for acc in history['train_acc']], 'o-', label='Train Accuracy', color='#2980b9')
axes[1].plot(epochs_range, [acc * 100 for acc in history['val_acc']], 's-', label='Val Accuracy', color='#27ae60')
axes[1].set_title('Training vs Validation Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

---
### 13. Held-out Test Set Evaluation
Load the best model checkpoint and evaluate exactly once on the held-out test set (15% of dataset).

**Key Medical Metric: Sensitivity / Recall**
$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
High recall is essential in medical screening to minimize False Negatives (un-flagged DR cases).

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

test_labels_list = []
test_preds_list = []
test_probs_list = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        test_labels_list.extend(labels.numpy())
        test_preds_list.extend(preds.cpu().numpy())
        test_probs_list.extend(probs[:, 1].cpu().numpy())

test_acc = accuracy_score(test_labels_list, test_preds_list)
test_precision = precision_score(test_labels_list, test_preds_list, zero_division=0)
test_recall = recall_score(test_labels_list, test_preds_list, zero_division=0)
test_f1 = f1_score(test_labels_list, test_preds_list, zero_division=0)
test_auc = roc_auc_score(test_labels_list, test_probs_list)

cm = confusion_matrix(test_labels_list, test_preds_list)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("==================================================")
print("           HELD-OUT TEST SET METRICS              ")
print("==================================================")
print(f"Accuracy:          {test_acc * 100:.2f}%")
print(f"Precision:         {test_precision * 100:.2f}%")
print(f"Recall/Sensitivity:{test_recall * 100:.2f}%  <-- Critical Screening Metric")
print(f"Specificity:       {specificity * 100:.2f}%")
print(f"F1-Score:          {test_f1:.4f}")
print(f"ROC-AUC:           {test_auc:.4f}")
print("--------------------------------------------------")
print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}  <-- Target to minimize")
print(f"True Positives (TP):  {tp}")
print("==================================================")

---
### 14. Confusion Matrix Visualization
Display heatmap of confusion matrix for test set predictions.

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=['Predicted No DR', 'Predicted DR'],
    yticklabels=['Actual No DR', 'Actual DR']
)
plt.title('Test Set Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

---
### 15. Classification Report
Display detailed per-class precision, recall, and F1-score report.

In [ ]:
report = classification_report(
    test_labels_list, 
    test_preds_list, 
    target_names=['No DR (0)', 'DR Present (1)']
)
print("Classification Report:\n")
print(report)

---
### 16. ROC Curve & ROC-AUC
Plot Receiver Operating Characteristic (ROC) curve using predicted DR probabilities.

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(test_labels_list, test_probs_list)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'EfficientNet-B0 (AUC = {test_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

---
### 17. Individual Image Prediction Function
Implement `predict_image(image_path)` to accept any retinal fundus image file and return confidence probabilities and responsible screening output.

In [ ]:
def predict_image(image_path, model, transform, device):
    """
    Performs inference on a single retinal fundus image.
    Returns prediction dictionary formatted for screening output.
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found at {image_path}")
        
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
        
    no_dr_prob = float(probs[0])
    dr_prob = float(probs[1])
    
    prediction_label = "DR PRESENT" if dr_prob >= 0.50 else "NO DR"
    confidence = dr_prob if prediction_label == "DR PRESENT" else no_dr_prob
    
    medical_recommendation = (
        "Possible diabetic retinopathy detected — specialist evaluation recommended."
        if prediction_label == "DR PRESENT"
        else "No signs of diabetic retinopathy detected."
    )
    
    return {
        "image_path": image_path,
        "prediction": prediction_label,
        "confidence_pct": round(confidence * 100, 2),
        "no_dr_probability": round(no_dr_prob, 4),
        "dr_probability": round(dr_prob, 4),
        "recommendation": medical_recommendation
    }

print("Inference function 'predict_image()' successfully created.")

---
### 18. Demonstration on Unseen Test Images
Test `predict_image()` on sample images from the held-out test set.

In [ ]:
# Pick sample No DR and DR Present images from test set
sample_no_dr = test_df[test_df['label'] == 0].iloc[0]
sample_dr    = test_df[test_df['label'] == 1].iloc[0]

test_samples = [sample_no_dr, sample_dr]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, sample in enumerate(test_samples):
    img_name = f"{sample['id_code']}.png"
    img_path = os.path.join(IMAGES_DIR, img_name)
    true_label = "NO DR" if sample['label'] == 0 else "DR PRESENT"
    
    res = predict_image(img_path, model, val_test_transforms, device)
    
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(
        f"True: {true_label}\n"
        f"Prediction: {res['prediction']} ({res['confidence_pct']}%)\n"
        f"No DR: {res['no_dr_probability']*100:.1f}% | DR: {res['dr_probability']*100:.1f}%",
        fontsize=10
    )
    axes[i].axis('off')
    
    print(f"=== Sample {i+1} Inference ===")
    print(f"Image: {img_name}")
    print(f"True Label: {true_label}")
    print(f"Prediction: {res['prediction']}")
    print(f"Confidence: {res['confidence_pct']}%")
    print(f"Probabilities: No DR={res['no_dr_probability']}, DR={res['dr_probability']}")
    print(f"Message: {res['recommendation']}\n")

plt.tight_layout()
plt.show()

---
### 19. Model Checkpointing & Download Package
Save trained model weights to `dr_efficientnet_b0.pth`, create `model_metadata.json`, and bundle into `medvision_dr_model.zip` for local download.

In [ ]:
OUTPUT_DIR = "/kaggle/working"
MODEL_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, "dr_efficientnet_b0.pth")
METADATA_PATH = os.path.join(OUTPUT_DIR, "model_metadata.json")
ZIP_PATH = os.path.join(OUTPUT_DIR, "medvision_dr_model.zip")

# Save PyTorch state dict
torch.save(model.state_dict(), MODEL_WEIGHTS_PATH)
print(f"Saved trained model weights to: {MODEL_WEIGHTS_PATH}")

# Metadata
metadata = {
    "project": "MedVisionAI",
    "model_architecture": "EfficientNet-B0",
    "pretrained_weights": "ImageNet DEFAULT",
    "task": "Binary Diabetic Retinopathy Screening",
    "classes": {
        "0": "No DR",
        "1": "DR Present"
    },
    "image_input_size": [224, 224, 3],
    "normalization": {
        "mean": IMAGENET_MEAN,
        "std": IMAGENET_STD
    },
    "dataset": "APTOS 2019 Blindness Detection",
    "test_metrics": {
        "accuracy": round(float(test_acc), 4),
        "precision": round(float(test_precision), 4),
        "recall": round(float(test_recall), 4),
        "specificity": round(float(specificity), 4),
        "f1_score": round(float(test_f1), 4),
        "roc_auc": round(float(test_auc), 4),
        "confusion_matrix": {
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp)
        }
    }
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Saved model metadata JSON to: {METADATA_PATH}")

# Create ZIP archive
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(MODEL_WEIGHTS_PATH, arcname=os.path.basename(MODEL_WEIGHTS_PATH))
    zipf.write(METADATA_PATH, arcname=os.path.basename(METADATA_PATH))

print(f"Successfully packaged export bundle: {ZIP_PATH}")
print(f"Archive Size: {os.path.getsize(ZIP_PATH) / 1e6:.2f} MB")
print("\nNotebook execution complete. Download 'medvision_dr_model.zip' from Kaggle output!")